RAG Pipeline


```
question ──► analyse (banque, année) ──► embedding ──► ChromaDB (top-k + filtre) ──► seuil 0,3
                                                                                      │
réponse + sources ◄── LLM (temperature 0) ◄── prompt = règles + extraits numérotés ◄──┘
```

**Entrée :** collection ChromaDB `bank_annual_reports_minilm` (dossier `models/chroma/`, notebook 02).
**Sorties :** `reports/rag_runs.jsonl` (toutes les exécutions), `reports/rag_results.csv` (résultats et métriques).

| Consigne du sujet | Section |
|---|---|
| 1. Prompt système (analyste conformité, citations, « je ne sais pas », format) | 3 |
| 2. Pipeline : question → embedding → top-k → prompt → LLM → réponse | 1, 2, 4, 5 |
| 3. 15 questions de conformité (5 banques, 5 catégories de risque) | 6 |
| 4. Enregistrer extraits, réponse, exactitude, citations | 5, 7, 8 |
| 5. Exactitude et exactitude des citations | 9 |
| 6. Modes d'échec : recherche ou génération ? | 9 et 10 |

**Sans clé OpenAI**, le notebook tourne en mode `DRY_RUN` : la recherche et la construction du prompt fonctionnent, mais le LLM n'est pas appelé. Avec une clé dans `module4_genai_bank_rag/.env`, il appelle `gpt-4o-mini` (environ 0,02 $ pour les 20 questions).

**Pour relancer :** Restart puis Run All. Les réponses déjà obtenues sont rechargées depuis `reports/rag_runs.jsonl`.

## 0. Configuration

In [1]:
import json, re, os, time, warnings
from pathlib import Path
import numpy as np
import pandas as pd

warnings.filterwarnings("ignore", category=FutureWarning)

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
MODELS, REPORTS = ROOT / "models", ROOT / "reports"
REPORTS.mkdir(exist_ok=True)

from dotenv import load_dotenv
load_dotenv(ROOT / ".env")

# Recherche (mêmes réglages que le notebook 02)
EMBED_MODEL, MAX_SEQ = "all-MiniLM-L6-v2", 512
COLLECTION = "bank_annual_reports_minilm"
K = 5                    # extraits donnés au LLM
K_PER_BANK = 3           # extraits par banque quand la question compare plusieurs banques
SIM_MIN = 0.3            # en dessous, on ne donne rien au LLM

# --- Génération ---
LLM_MODEL = "gpt-4o-mini"
TEMPERATURE = 0          # conformité : même question, même réponse
MAX_OUTPUT_TOKENS = 700
SEED = 42
PRICE_IN, PRICE_OUT = 0.15, 0.60      # $ par million de tokens
PROMPT_VERSION = "v1"    # à changer à chaque modification du prompt : c'est la clé du cache

HAS_KEY = os.getenv("OPENAI_API_KEY", "").startswith("sk-")
DRY_RUN = not HAS_KEY
print("Clé OpenAI détectée :", HAS_KEY, "| mode :", "DRY_RUN (le LLM n'est pas appelé)" if DRY_RUN else f"LLM réel ({LLM_MODEL}, temperature={TEMPERATURE})")

Clé OpenAI détectée : False | mode : DRY_RUN (le LLM n'est pas appelé)


In [2]:
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings

cfg = json.loads((MODELS / "minilm_config.json").read_text())
assert cfg["model"] == EMBED_MODEL and cfg["max_seq_length"] == MAX_SEQ, "Le modèle ou la longueur max diffèrent du notebook 02 : les vecteurs ne seraient plus comparables."

embedder = SentenceTransformer(EMBED_MODEL, device="cpu")
embedder.max_seq_length = MAX_SEQ

client = chromadb.PersistentClient(path=str(MODELS / "chroma"), settings=Settings(anonymized_telemetry=False))
col = client.get_collection(COLLECTION)
assert col.count() == cfg["n_chunks"], f"{col.count()} chunks dans la collection, {cfg['n_chunks']} attendus : relancer le notebook 02."
print(f"Collection '{COLLECTION}' : {col.count()} chunks | modèle d'embedding : {EMBED_MODEL}")

llm = None
if HAS_KEY:
    try:
        from openai import OpenAI
        llm = OpenAI()
    except ImportError:
        DRY_RUN = True
        print("Package openai absent : pip install openai, puis Restart. Mode DRY_RUN pour l'instant.")

/Users/carchaf/jarvis_data_eng_SamiaCarchaf/module4_genai_bank_rag/.venv/lib/python3.11/site-packages/sentence_transformers/cross_encoder/CrossEncoder.py:11: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from tqdm.autonotebook import tqdm, trange


Collection 'bank_annual_reports_minilm' : 3562 chunks | modèle d'embedding : all-MiniLM-L6-v2


## 1. Recherche

On reprend les fonctions du notebook 02 (elles iront dans `src/` plus tard). Trois règles issues de ce qu'on a mesuré :
- **La banque et l'année viennent d'un filtre de métadonnées**, pas du texte de la question.
- **On enlève le nom de la banque de la requête** (« the bank » à la place) : la précision était meilleure sans (notebook 02, v2).
- **Une banque absente de la base est refusée avant la recherche** : le seuil de similarité ne la détectait pas (Desjardins : 0,408).

In [ ]:
def build_where(bank=None, year=None):
    conds = []
    if bank: conds.append({"bank_code": bank})
    if year: conds.append({"fiscal_year": year})
    if not conds: return None
    return conds[0] if len(conds) == 1 else {"$and": conds}

def embed_query(q):
    return embedder.encode([q], normalize_embeddings=True, convert_to_numpy=True)[0]

def retrieve_topk(q, k=K, where=None):
    res = col.query(query_embeddings=[embed_query(q).tolist()], n_results=k, where=where,
                    include=["documents", "metadatas", "distances"])
    return [{"chunk_id": cid, "sim": round(1 - d, 4), "citation": m["citation"], "bank_code": m["bank_code"],
             "fiscal_year": m["fiscal_year"], "section": m["section"], "risk_category": m["risk_category"],
             "pages": f'{m["page_start"]}-{m["page_end"]}', "text": doc}
            for cid, doc, m, d in zip(res["ids"][0], res["documents"][0], res["metadatas"][0], res["distances"][0])]

# Analyse de la question : banque(s), année(s), hors périmètre
BANKS = {
    "RBC":  r"Royal Bank of Canada|\bRBC\b|Royal Bank",
    "TD":   r"Toronto[- ]Dominion|\bTD\b",
    "BMO":  r"Bank of Montreal|\bBMO\b",
    "BNS":  r"Bank of Nova Scotia|Scotiabank|\bBNS\b|Scotia",
    "CIBC": r"Canadian Imperial|\bCIBC\b",
}
NOT_IN_BASE = r"Desjardins|National Bank|Laurentian|HSBC Canada|Tangerine|EQ Bank|Canadian Western|Manulife Bank"
SUPPORTED_YEARS = {2024, 2025}

def analyze_question(q):
    banks = [c for c, p in BANKS.items() if re.search(p, q, re.I)]
    years = sorted({int(y) for y in re.findall(r"(?<!\d)(20\d{2})(?!\d)", q)})
    unknown = re.search(NOT_IN_BASE, q, re.I)
    return {"banks": banks, "years": [y for y in years if y in SUPPORTED_YEARS],
            "unsupported_years": [y for y in years if y not in SUPPORTED_YEARS],
            "unknown_bank": unknown.group(0) if unknown else None}

def strip_bank_names(q):
    for p in BANKS.values():
        q = re.sub(rf"(?:{p})(?P<poss>[’']s)?", lambda m: "the bank" + (m.group("poss") or ""), q, flags=re.I)
    return re.sub(r"the bank and the bank", "the banks", q)

def retrieve(question, info):
    sq = strip_bank_names(question)
    year = info["years"][0] if len(info["years"]) == 1 else None
    if len(info["banks"]) > 1:
        out = []
        for b in info["banks"]:
            out += retrieve_topk(sq, K_PER_BANK, build_where(b, year))
        return sq, out
    bank = info["banks"][0] if info["banks"] else None
    return sq, retrieve_topk(sq, K, build_where(bank, year))

## 2. Contexte envoyé au LLM

Chaque extrait reçoit un numéro `[S1]`, `[S2]`… avec sa citation (banque, document, section, pages). Le LLM doit citer ces numéros : on peut ensuite **vérifier automatiquement** qu'il n'invente pas de source.

In [4]:
def build_context(chunks):
    blocks = [f"[S{i}] {c['citation']}\n{c['text'].strip()}" for i, c in enumerate(chunks, 1)]
    return "\n\n---\n\n".join(blocks)

def build_user_prompt(question, chunks):
    return (f"Question: {question}\n\n"
            f"Context (numbered excerpts from the annual reports):\n\n{build_context(chunks)}\n\n"
            "Answer the question using only the excerpts above, following the rules and format given.")

## 3. Prompt système (consigne 1)

Il applique les bonnes pratiques du cours : **rôle** d'analyste conformité, **citations obligatoires**, phrase **« je ne sais pas »** imposée, **exemples (few-shot)** pour les cas limites (information partielle, données contradictoires, hors périmètre), **format de sortie** fixe. Les exemples utilisent une banque fictive « Example Bank » pour ne jamais injecter de vrais chiffres.

Pour l'améliorer : **changer une seule chose à la fois**, changer `PROMPT_VERSION`, relancer, comparer les métriques de la section 9.

In [5]:
REFUSAL = "The provided filings do not contain enough information to answer this question."

SYSTEM_PROMPT = f"""You are a compliance analyst specialised in Canadian bank disclosures. You answer questions using ONLY numbered excerpts from the annual reports of RBC, TD, BMO, Scotiabank and CIBC for fiscal years 2024 and 2025.

RULES
1. Use only the excerpts provided. Never use outside knowledge. Never speculate, estimate or forecast.
2. Cite the source of every claim with its excerpt number in brackets, for example [S2]. At the end, list each cited source with bank name, document, section, fiscal year and pages.
3. If the excerpts do not contain the information needed, answer exactly: "{REFUSAL}" and do not add anything else.
4. If the excerpts only partly answer the question, give what is supported, then state clearly what is missing.
5. If excerpts from different filings give different values (for example fiscal 2024 and 2025), report each value with its fiscal year. Do not pick one silently.
6. Copy numbers exactly as written, with their units, the period and the bank they belong to. Never mix figures of different banks or years.
7. Simple arithmetic is allowed only on numbers present in the excerpts: show the calculation and cite the excerpts. If a needed number is missing, say which one.
8. If the question is outside the scope of the excerpts (another bank, a year other than 2024 or 2025, a future event, a non-banking topic), apply rule 3.

OUTPUT FORMAT
Answer: <two to six sentences or short bullets, with [S#] after each claim>
Sources:
- [S#] <Bank>, <document>, <section>, FY<year>, pp.<pages>

EXAMPLES (illustrative, fictional bank)
Q: What was Example Bank's CET1 ratio at year-end?  Excerpts: [S1] says the ratio "increased 0.5 percentage points" but never gives the level.
A: Answer: The excerpts say that Example Bank's CET1 ratio increased by 0.5 percentage points [S1], but they do not state its level at year-end.
Sources:
- [S1] Example Bank, Annual Report, MD&A, FY2025, pp.10-11

Q: What was Example Bank's liquidity coverage ratio?  Excerpts: [S1] (FY2024) says 125%, [S2] (FY2025) says 131%.
A: Answer: The ratio was 125% in fiscal 2024 [S1] and 131% in fiscal 2025 [S2].
Sources:
- [S1] Example Bank, Annual Report, Liquidity risk, FY2024, pp.40
- [S2] Example Bank, Annual Report, Liquidity risk, FY2025, pp.42

Q: What will Example Bank's CET1 ratio be in 2027?
A: {REFUSAL}"""
print(SYSTEM_PROMPT[:600] + "\n[...]")

You are a compliance analyst specialised in Canadian bank disclosures. You answer questions using ONLY numbered excerpts from the annual reports of RBC, TD, BMO, Scotiabank and CIBC for fiscal years 2024 and 2025.

RULES
1. Use only the excerpts provided. Never use outside knowledge. Never speculate, estimate or forecast.
2. Cite the source of every claim with its excerpt number in brackets, for example [S2]. At the end, list each cited source with bank name, document, section, fiscal year and pages.
3. If the excerpts do not contain the information needed, answer exactly: "The provided filing
[...]


## 4. Appel du LLM

`temperature=0` et une graine fixe : la même question redonne la même réponse (exigence de conformité). Le coût est calculé à partir des tokens réellement facturés. En `DRY_RUN`, rien n'est envoyé.

In [6]:
def call_llm(system, user):
    if DRY_RUN:
        return {"answer": "[DRY_RUN] LLM non appelé.", "tokens_in": 0, "tokens_out": 0, "cost": 0.0, "seconds": 0.0}
    last = None
    for attempt in range(4):
        try:
            t0 = time.time()
            r = llm.chat.completions.create(model=LLM_MODEL, temperature=TEMPERATURE, seed=SEED, max_tokens=MAX_OUTPUT_TOKENS,
                                            messages=[{"role": "system", "content": system}, {"role": "user", "content": user}])
            u = r.usage
            return {"answer": r.choices[0].message.content.strip(), "tokens_in": u.prompt_tokens, "tokens_out": u.completion_tokens,
                    "cost": u.prompt_tokens / 1e6 * PRICE_IN + u.completion_tokens / 1e6 * PRICE_OUT, "seconds": round(time.time() - t0, 2)}
        except Exception as e:
            last = e
            time.sleep(2 ** attempt)
    raise RuntimeError(f"Échec de l'appel LLM après 4 essais : {last}")

## 5. Le pipeline

Ordre : analyse de la question → **gardes** (banque absente de la base, année hors 2024-2025) → recherche → **seuil** → LLM. Les gardes répondent sans appeler le LLM (rapide, gratuit, déterministe). `guards=False` les désactive pour tester si le **prompt seul** sait refuser (section 6).

Chaque exécution est enregistrée avec la question, les extraits récupérés, la réponse, les tokens et le coût.

In [7]:
RUNS_PATH = REPORTS / "rag_runs.jsonl"
RUNS = {}                                  # clé : (qid, PROMPT_VERSION, LLM_MODEL)
if RUNS_PATH.exists():
    for line in RUNS_PATH.read_text().splitlines():
        r = json.loads(line)
        RUNS[(r["qid"], r["prompt_version"], r["llm_model"])] = r

def cited_numbers(answer):
    return sorted({int(n) for n in re.findall(r"\[S(\d+)\]", answer)})

def rag_answer(qid, question, guards=True, use_cache=True):
    key = (qid, PROMPT_VERSION, LLM_MODEL)
    if use_cache and key in RUNS and not DRY_RUN:
        return RUNS[key]
    info = analyze_question(question)
    base = {"qid": qid, "question": question, "prompt_version": PROMPT_VERSION, "llm_model": LLM_MODEL, "guards": guards,
            "filters": {"banks": info["banks"], "years": info["years"]}, "search_query": None, "chunks": [],
            "tokens_in": 0, "tokens_out": 0, "cost": 0.0, "seconds": 0.0}
    if guards and info["unknown_bank"]:
        return {**base, "route": "guard_bank", "answer": REFUSAL, "refused": True}
    if guards and info["unsupported_years"]:
        return {**base, "route": "guard_year", "answer": REFUSAL, "refused": True}
    sq, found = retrieve(question, info)
    kept = [c for c in found if c["sim"] >= SIM_MIN]
    base.update(search_query=sq, chunks=kept)
    if not kept:
        return {**base, "route": "below_threshold", "answer": REFUSAL, "refused": True}
    out = call_llm(SYSTEM_PROMPT, build_user_prompt(question, kept))
    res = {**base, **out, "route": "llm" if not DRY_RUN else "dry_run",
           "refused": REFUSAL.lower() in out["answer"].lower()}
    cites = cited_numbers(out["answer"])
    res["cited"] = cites
    res["citations_valid"] = bool(cites) and all(1 <= n <= len(kept) for n in cites)   # contrôle automatique : numéros existants
    if not DRY_RUN:
        RUNS[key] = res
    return res

def save_runs():
    RUNS_PATH.write_text("\n".join(json.dumps(r, ensure_ascii=False) for r in RUNS.values()))

# Test rapide du routage (aucun appel LLM pour les trois premiers)
for q in ["What is Desjardins' CET1 ratio?", "What will RBC's CET1 ratio be in 2027?", "What is the capital of France?"]:
    r = rag_answer("test", q, use_cache=False)
    print(f"{q!r:55} -> {r['route']}")
print("\nAnalyse :", analyze_question("Compare how RBC and TD describe liquidity risk in fiscal 2025."))
print("Requête nettoyée :", strip_bank_names("What was TD's Liquidity Coverage Ratio at October 31, 2025?"))

"What is Desjardins' CET1 ratio?"                       -> guard_bank
"What will RBC's CET1 ratio be in 2027?"                -> guard_year
'What is the capital of France?'                        -> below_threshold

Analyse : {'banks': ['RBC', 'TD'], 'years': [2025], 'unsupported_years': [], 'unknown_bank': None}
Requête nettoyée : What was the bank's Liquidity Coverage Ratio at October 31, 2025?


## 6. Les questions de test (consigne 3)

**15 questions de conformité** : 5 banques, 5 catégories de risque, 2 années, avec 3 questions à chiffre et 1 comparaison entre banques. Puis **5 questions pièges** (bonne pratique du cours) : banque absente, événement futur, hors sujet, année non couverte, calcul.

Les questions sont fixées **avant** de voir les réponses. Aucun résultat attendu n'est écrit ici : l'exactitude se juge en relisant les extraits (section 8).

In [8]:
QUESTIONS = [
    # (id, catégorie de risque, question)
    (1,  "Credit",      "How does RBC describe the main drivers of its provision for credit losses in fiscal 2025?"),
    (2,  "Credit",      "What does TD say about its allowance for credit losses at October 31, 2024?"),
    (3,  "Credit",      "How does CIBC manage credit risk in its commercial lending portfolio in fiscal 2025?"),
    (4,  "Market",      "How does BMO measure and manage market risk, including Value at Risk, in fiscal 2025?"),
    (5,  "Market",      "How sensitive is TD's economic value of equity to interest rate changes in fiscal 2025?"),
    (6,  "Climate",     "What does Scotiabank say about climate-related risks and its net-zero commitments in fiscal 2024?"),
    (7,  "Climate",     "How does RBC describe its approach to climate risk in fiscal 2024?"),
    (8,  "Climate",     "What does CIBC disclose about climate-related risk management in fiscal 2024?"),
    (9,  "Operational", "How does BMO describe its management of operational risk, including technology and cyber risk, in fiscal 2024?"),
    (10, "Operational", "What does TD say about third-party risk management in fiscal 2025?"),
    (11, "Operational", "How does Scotiabank describe its use of artificial intelligence and the related risks in fiscal 2025?"),
    (12, "Liquidity",   "What was TD's Liquidity Coverage Ratio for the quarter ended October 31, 2025?"),
    (13, "Liquidity",   "How does RBC manage liquidity risk in fiscal 2024?"),
    (14, "Liquidity",   "How does CIBC describe its liquidity risk management framework in fiscal 2025?"),
    (15, "Liquidity",   "Compare how RBC and TD describe their liquidity risk management in fiscal 2025."),
]
ADVERSARIAL = [
    ("A1", "Banque absente",  "What is Desjardins' CET1 ratio at October 31, 2025?"),
    ("A2", "Événement futur", "What will RBC's CET1 ratio be at the end of fiscal 2027?"),
    ("A3", "Hors sujet",      "What is the capital of France?"),
    ("A4", "Année non couverte", "What was BMO's provision for credit losses in fiscal 2023?"),
    ("A5", "Calcul",          "Calculate the average of RBC's and TD's CET1 capital ratios at October 31, 2025."),
]
MAIN_IDS = [q[0] for q in QUESTIONS]
CAT = {q[0]: q[1] for q in QUESTIONS}
Q_TEXT = {q[0]: q[2] for q in QUESTIONS} | {q[0]: q[2] for q in ADVERSARIAL}

cover = pd.DataFrame([{"Q": i, "catégorie": c, "banques": ", ".join(analyze_question(t)["banks"]), "années": analyze_question(t)["years"]} for i, c, t in QUESTIONS])
print(cover.to_string(index=False))
print("\nBanques couvertes :", sorted({b for r in cover["banques"] for b in r.split(", ") if b}), "| catégories :", sorted(set(CAT.values())))

 Q   catégorie banques années
 1      Credit     RBC [2025]
 2      Credit      TD [2024]
 3      Credit    CIBC [2025]
 4      Market     BMO [2025]
 5      Market      TD [2025]
 6     Climate     BNS [2024]
 7     Climate     RBC [2024]
 8     Climate    CIBC [2024]
 9 Operational     BMO [2024]
10 Operational      TD [2025]
11 Operational     BNS [2025]
12   Liquidity      TD [2025]
13   Liquidity     RBC [2024]
14   Liquidity    CIBC [2025]
15   Liquidity RBC, TD [2025]

Banques couvertes : ['BMO', 'BNS', 'CIBC', 'RBC', 'TD'] | catégories : ['Climate', 'Credit', 'Liquidity', 'Market', 'Operational']


## 7. Exécution des 15 questions

En `DRY_RUN`, la colonne `route` vaut `dry_run` : la recherche a eu lieu, le LLM non. Avec une clé, chaque réponse est enregistrée dans `reports/rag_runs.jsonl` ; relancer la cellule ne rappelle pas le LLM pour ce qui est déjà fait.

In [9]:
main = {}
for qid, cat, q in QUESTIONS:
    main[qid] = rag_answer(qid, q)
if not DRY_RUN:
    save_runs()

tbl = pd.DataFrame([{"Q": i, "catégorie": CAT[i], "route": r["route"], "extraits": len(r["chunks"]),
                     "sim max": max((c["sim"] for c in r["chunks"]), default=0),
                     "banques": ",".join(sorted({c["bank_code"] for c in r["chunks"]})),
                     "années": ",".join(str(y) for y in sorted({c["fiscal_year"] for c in r["chunks"]})),
                     "refus": r["refused"], "citations valides (auto)": r.get("citations_valid"),
                     "tokens": r["tokens_in"] + r["tokens_out"], "coût $": round(r["cost"], 4)} for i, r in main.items()])
print(tbl.to_string(index=False))
print(f"\nCoût total : {sum(r['cost'] for r in main.values()):.4f} $ | tokens : {sum(r['tokens_in'] + r['tokens_out'] for r in main.values()):,}")

 Q   catégorie   route  extraits  sim max banques années  refus  citations valides (auto)  tokens  coût $
 1      Credit dry_run         5   0.6060     RBC   2025  False                     False       0     0.0
 2      Credit dry_run         5   0.5957      TD   2024  False                     False       0     0.0
 3      Credit dry_run         5   0.5706    CIBC   2025  False                     False       0     0.0
 4      Market dry_run         5   0.5479     BMO   2025  False                     False       0     0.0
 5      Market dry_run         5   0.5182      TD   2025  False                     False       0     0.0
 6     Climate dry_run         5   0.7002     BNS   2024  False                     False       0     0.0
 7     Climate dry_run         5   0.5339     RBC   2024  False                     False       0     0.0
 8     Climate dry_run         5   0.6415    CIBC   2024  False                     False       0     0.0
 9 Operational dry_run         5   0.6350     

### Questions pièges, avec et sans gardes

- **Avec gardes** : les règles de la section 5 refusent avant le LLM.
- **Sans gardes** : le LLM reçoit quand même des extraits, et c'est **le prompt seul** qui doit refuser. C'est le vrai test du prompt.

In [11]:
adv = {}
for qid, kind, q in ADVERSARIAL:
    adv[qid] = rag_answer(qid, q)
    adv[qid + "-noguard"] = rag_answer(qid + "-noguard", q, guards=False)
if not DRY_RUN:
    save_runs()

kinds = {a[0]: a[1] for a in ADVERSARIAL}
rows = [{"Q": k.split("-")[0], "type": kinds[k.split("-")[0]], "gardes": "non" if k.endswith("noguard") else "oui",
         "route": r["route"], "refus": r["refused"]} for k, r in adv.items()]
print(pd.DataFrame(rows).sort_values(["Q", "gardes"], ascending=[True, False]).to_string(index=False))
print("\nA5 (calcul) : pas de refus attendu a priori. Le bon comportement est de calculer avec des chiffres cités, ou de dire quel chiffre manque. À relire à la main.")

 Q               type gardes           route  refus
A1     Banque absente    oui      guard_bank   True
A1     Banque absente    non         dry_run  False
A2    Événement futur    oui      guard_year   True
A2    Événement futur    non         dry_run  False
A3         Hors sujet    oui below_threshold   True
A3         Hors sujet    non below_threshold   True
A4 Année non couverte    oui      guard_year   True
A4 Année non couverte    non         dry_run  False
A5             Calcul    oui         dry_run  False
A5             Calcul    non         dry_run  False

A5 (calcul) : pas de refus attendu a priori. Le bon comportement est de calculer avec des chiffres cités, ou de dire quel chiffre manque. À relire à la main.


## 8. Relecture manuelle (consignes 4 et 5)

Pour chaque question : lire la réponse, puis **vérifier dans les extraits** (et au besoin dans le PDF) :
1. **La réponse est-elle exacte ?** `True` (correcte et complète), `"partial"` (correcte mais incomplète, ou un détail faux), `False`.
2. **Les citations sont-elles exactes ?** Chaque `[S#]` pointe-t-il vers un extrait qui contient bien ce qui est affirmé ? `True` / `False`.
3. **Si ce n'est pas `True` : où est la panne ?** `"retrieval"` (les extraits ne contenaient pas la réponse), `"generation"` (les extraits la contenaient, le LLM a mal répondu) ou `"both"`.

`show_runs([1, 2, 3])` affiche les réponses et les extraits ; `grep_chunk` cherche un mot dans un extrait.

In [12]:
def show_runs(ids, width=350):
    for qid in ids:
        r = main.get(qid) or adv.get(qid)
        print("=" * 110)
        print(f"Q{qid} [{CAT.get(qid, '')}] {r['question']}")
        print(f"filtres : {r['filters']} | requête : {r['search_query']} | route : {r['route']}")
        print("-" * 110)
        print(r["answer"])
        print("-" * 110)
        for i, c in enumerate(r["chunks"], 1):
            print(f"  [S{i}] sim {c['sim']:.2f} | {c['chunk_id']} | {c['citation'][:70]}")
            print("       " + " ".join(c["text"].split())[:width] + "…")

def grep_chunk(qid, n, pattern, before=150, after=300):
    r = main.get(qid) or adv.get(qid)
    t = r["chunks"][n - 1]["text"]
    ms = list(re.finditer(pattern, t, re.I))
    print(f"Q{qid} [S{n}] {r['chunks'][n - 1]['chunk_id']} : {len(ms)} occurrence(s) de /{pattern}/")
    for m in ms[:4]:
        s = max(0, m.start() - before)
        print("   …" + " ".join(t[s:m.end() + after].split()) + "…")

show_runs([1, 2, 3])        # changer la liste pour relire les autres questions

Q1 [Credit] How does RBC describe the main drivers of its provision for credit losses in fiscal 2025?
filtres : {'banks': ['RBC'], 'years': [2025]} | requête : How does the bank describe the main drivers of its provision for credit losses in fiscal 2025? | route : dry_run
--------------------------------------------------------------------------------------------------------------
[DRY_RUN] LLM non appelé.
--------------------------------------------------------------------------------------------------------------
  [S1] sim 0.61 | RBC_annual_report_FY2025-0235 | Royal Bank of Canada – Annual Report FY2025 – Note 5 Loans and allowan
       • Maturities, which reflect the allowance related to assets derecognized during the period without a credit loss being incurred, including those assets that were derecognized following a modification of terms. • Changes in risk, parameters and exposures, which comprise the impact of changes in model inputs or assumptions, including changes in forwa…

In [13]:
show_runs([3], width=1500)

Q3 [Credit] How does CIBC manage credit risk in its commercial lending portfolio in fiscal 2025?
filtres : {'banks': ['CIBC'], 'years': [2025]} | requête : How does the bank manage credit risk in its commercial lending portfolio in fiscal 2025? | route : dry_run
--------------------------------------------------------------------------------------------------------------
[DRY_RUN] LLM non appelé.
--------------------------------------------------------------------------------------------------------------
  [S1] sim 0.57 | CIBC_annual_report_FY2025-0127 | Canadian Imperial Bank of Commerce – Annual Report FY2025 – Liquidity 
       (1) Includes non-negotiable term deposits from banks. (2) Includes wholesale funding liabilities which are subject to conversion under bail-in regulations. See the “Capital management” section for additional details. (3) Includes $500 million (2024: $500 million) of HELOC securitization. (4) Includes Federal Home Loan Bank (FHLB) deposits. The following tabl

- S1 : financement et liquidité, hors sujet.
- S2 et S4 : disent que le groupe « Global Credit Risk Management » est chargé de l’octroi et de la surveillance des risques de crédit des portefeuilles commerciaux. C’est partiel : on apprend qui s’en occupe, pas comment (limites, analyse des dossiers, suivi, garanties).
- S3 : tableau réglementaire des expositions, non pertinent.
- S5 : glossaire, non pertinent.

##### **Pourquoi ça arrive ?**
La question est générale (« how does the bank manage credit risk in commercial lending ») et beaucoup de chunks parlent de risque en termes proches : gouvernance, liquidité, trading. Les similarités sont presque plates (0,57 à 0,54), donc la recherche ne les départage pas.

--> Un test à faire : ajouter le filtre risk_category
Dans le notebook 02, on avait vu que ce champ existe (« Credit risk »). Si on filtre dessus, la recherche se limite aux chunks de crédit.

In [14]:
q = strip_bank_names(Q_TEXT[3])
where = {"$and": [{"bank_code": "CIBC"}, {"fiscal_year": 2025}, {"risk_category": "Credit risk"}]}
for i, c in enumerate(retrieve_topk(q, 5, where), 1):
    print(f"[S{i}] sim {c['sim']:.2f} | {c['chunk_id']} | {c['section'][:50]}")
    print("     " + " ".join(c["text"].split())[:500] + "…")

[S1] sim 0.56 | CIBC_annual_report_FY2025-0088 | Credit Risk
     Capital Markets Risk Management: This group is responsible for independent oversight of the measurement, monitoring and control of traded and non-traded market risk, liquidity risk and trading credit risk (including CVA risk), including adjudication of trading credit facilities for banks, non-bank financial entities, prime brokerage clients and central clearing counterparties. In addition, CMRM is responsible for the risk management of sovereign and country risk, securitizations and the oversigh…
[S2] sim 0.55 | CIBC_annual_report_FY2025-0307 | Note 26 Financial instruments – disclosures
     The table below sets out the categories of the on-balance sheet exposures that are subject to the credit risk framework as set out in the CAR Guideline issued by OSFI under the different Basel approaches based on the carrying value of those exposures in our consolidated financial statements. The credit risk framework includes CCR ex

Le filtre n’a rien réglé. On retrouve presque les mêmes chunks (0088, 0307, 0075) et deux autres (0097, 0087) qui parlent d’expositions réglementaires et de profil de risque général. Aucun ne décrit comment CIBC gère le risque de crédit commercial. Le filtre de catégorie n’est donc pas la solution, et tu peux l’écarter.

On ne sait pas encore pourquoi. Il y a deux possibilités :

Le passage existe, mais la recherche le classe loin (comme les chunks CIBC du notebook 02, rangs 20 à 258).
Le passage n’existe pas comme tel dans le rapport, ou il est noyé dans un tableau.

* Chercher dans tous les chunks CIBC 2025 ceux qui contiennent des mots comme « adjudicat », « underwrit », « collateral » ou « monitor », et donne leur rang dans la recherche :

In [15]:
where = {"$and": [{"bank_code": "CIBC"}, {"fiscal_year": 2025}]}
allc = col.get(where=where, include=["documents", "metadatas"])
n = len(allc["ids"])
ranked = col.query(query_embeddings=[embed_query(strip_bank_names(Q_TEXT[3])).tolist()],
                   n_results=n, where=where, include=["distances"])
rank = {cid: i + 1 for i, cid in enumerate(ranked["ids"][0])}
pat = re.compile(r"commercial (lending|credit|loan|portfolio)|adjudicat|underwrit|credit limit|collateral|early warning|watchlist|monitor", re.I)

rows = []
for cid, doc, m in zip(allc["ids"], allc["documents"], allc["metadatas"]):
    hits = len(pat.findall(doc))
    if hits >= 4:
        rows.append((hits, rank[cid], cid, m["section"][:40], doc))
rows.sort(reverse=True)
print(f"{n} chunks CIBC 2025 | {len(rows)} chunks avec au moins 4 mots-clés\n")
for hits, r, cid, sec, doc in rows[:6]:
    mt = pat.search(doc)
    s = max(0, mt.start() - 100)
    print(f"{cid} | mots-clés : {hits} | rang dans la recherche : {r}/{n} | {sec}")
    print("    …" + " ".join(doc[s:s + 400].split()) + "…\n")

332 chunks CIBC 2025 | 36 chunks avec au moins 4 mots-clés

CIBC_annual_report_FY2025-0089 | mots-clés : 14 | rang dans la recherche : 8/332 | Credit Risk
    …duce counterparty risk. Credit risk mitigation We may mitigate credit risk by obtaining a pledge of collateral, which improves recoveries in the event of a default. Our credit risk management policies include verification of the collateral and its value and ensuring that we have legal certainty with respect to the assets pledged. Valuations are updated periodically depending on the nature of the c…

CIBC_annual_report_FY2025-0214 | mots-clés : 10 | rang dans la recherche : 169/332 | Note 2 Fair value measurement
    …In order to reflect the observed market practice of pricing collateralized and uncollateralized derivatives, our valuation approach uses overnight indexed swap (OIS) curves as the discount rate for valuing collateralized derivatives and uses an estimated market cost of funds curve as the discount rate for valuing un

#### Ce qu’on voit

Un chunk pertinent existe : CIBC_annual_report_FY2025-0089, section « Credit Risk », qui décrit comment CIBC réduit son risque de crédit (garanties, vérification de leur valeur, certitude juridique). Il est au rang 8 sur 332, juste sous le top-5 que reçoit le LLM.
Le reste de la liste est du bruit.

In [17]:
GRADES = {}

In [18]:
GRADES[3] = dict(correct="partial", citations=None, failure="retrieval", note="chunk utile FY2025-0089 (réduction du risque de crédit) au rang 8, hors top-5; top-5 = gouvernance et tableaux")

## 9. Métriques (consignes 5 et 6)

In [19]:
graded = [i for i in MAIN_IDS if i in GRADES]
todo = [i for i in MAIN_IDS if i not in GRADES]
print(f"{len(graded)} / {len(MAIN_IDS)} questions jugées" + (f" | à juger : {todo}" if todo else ""))

if graded:
    g = pd.DataFrame([{"Q": i, "catégorie": CAT[i], "banque": ",".join(main[i]["filters"]["banks"]) or "toutes", **GRADES[i]} for i in graded])
    g["exacte (strict)"] = g.correct.map(lambda x: x is True)
    g["exacte (large)"] = g.correct.map(lambda x: x is True or x == "partial")
    g["citations ok"] = g.citations.map(lambda x: x is True)
    print(f"\nExactitude stricte : {g['exacte (strict)'].mean():.0%} ({g['exacte (strict)'].sum()}/{len(g)})")
    print(f"Exactitude large (partiel compte) : {g['exacte (large)'].mean():.0%} ({g['exacte (large)'].sum()}/{len(g)})")
    print(f"Citations exactes : {g['citations ok'].mean():.0%} ({g['citations ok'].sum()}/{len(g)})")
    print("\nPar catégorie de risque :")
    print(g.groupby("catégorie")[["exacte (strict)", "exacte (large)", "citations ok"]].mean().round(2).to_string())
    print("\nPar banque :")
    print(g.groupby("banque")[["exacte (strict)", "exacte (large)", "citations ok"]].mean().round(2).to_string())
    fails = g[g.failure.notna()]
    print("\nModes d'échec :", fails.failure.value_counts().to_dict() or "aucun")
    print(fails[["Q", "catégorie", "correct", "failure", "note"]].to_string(index=False))
else:
    print("Aucun verdict pour l'instant : remplir GRADES (section 8), puis relancer cette cellule.")

1 / 15 questions jugées | à juger : [1, 2, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15]

Exactitude stricte : 0% (0/1)
Exactitude large (partiel compte) : 100% (1/1)
Citations exactes : 0% (0/1)

Par catégorie de risque :
           exacte (strict)  exacte (large)  citations ok
catégorie                                               
Credit                 0.0             1.0           0.0

Par banque :
        exacte (strict)  exacte (large)  citations ok
banque                                               
CIBC                0.0             1.0           0.0

Modes d'échec : {'retrieval': 1}
 Q catégorie correct   failure                                                                                                           note
 3    Credit partial retrieval chunk utile FY2025-0089 (réduction du risque de crédit) au rang 8, hors top-5; top-5 = gouvernance et tableaux


## 10. Analyse des échecs et conclusions (consigne 6)

*À compléter après la relecture, avec les chiffres de la section 9. Rien n'est pré-rempli : les résultats ne sont connus qu'une fois les 15 réponses relues.*

### Résultats
- Exactitude (stricte / large) : …
- Citations exactes : …
- Coût et temps : … tokens, … $, … s par question.

### Où est la panne : recherche ou génération ?
Pour chaque question jugée fausse ou partielle : les extraits contenaient-ils la réponse ?
- **Panne de recherche** : …
- **Panne de génération** : …

### Questions pièges
- Avec gardes : … / Sans gardes (prompt seul) : … Le LLM a-t-il refusé de lui-même ? A5 (calcul) : …

### Ce que le notebook 02 laissait prévoir
Les questions à chiffre avaient un precision@3 de 0,2. Les réponses à chiffre de cette étape confirment-elles ce faible score ? (Q12 est la question à chiffre dont on connaît l'extrait : TD FY2025-0182 contient le LCR de 130 %.)

### Limites
- 15 questions seulement ; jugement manuel d'une seule personne.
- Les gardes (banque absente, année) sont des règles à mots-clés, pas de la compréhension.
- Un seul prompt testé (v1) : une amélioration se mesure en changeant une chose à la fois.

### Décisions pour la suite
1. Notebook 04 : …
2. Pistes : recherche hybride pour les chiffres, re-ranking, extraction des tableaux.

In [ ]:
# Sauvegarde des résultats
rows = []
for i in MAIN_IDS:
    r, gr = main[i], GRADES.get(i, {})
    rows.append({"Q": i, "catégorie": CAT[i], "question": r["question"], "route": r["route"], "extraits": ";".join(c["chunk_id"] for c in r["chunks"]),
                 "sim max": max((c["sim"] for c in r["chunks"]), default=0), "réponse": r["answer"],
                 "correct": gr.get("correct"), "citations": gr.get("citations"), "failure": gr.get("failure"), "note": gr.get("note"),
                 "tokens": r["tokens_in"] + r["tokens_out"], "coût $": round(r["cost"], 5), "prompt": PROMPT_VERSION, "modèle LLM": LLM_MODEL})
pd.DataFrame(rows).to_csv(REPORTS / "rag_results.csv", index=False)
print("reports/rag_results.csv | reports/rag_runs.jsonl" + (" (aucune exécution LLM réelle : DRY_RUN)" if DRY_RUN else ""))